# Food Access Analysis — Washington State

This notebook explores food access and socioeconomic indicators across Washington State census tracts. It draws from three data sources: USDA food access data, ACS 5-year Census estimates, and TIGER/Line census tract shapefiles.

## **Data Vintage Note**

The data used within this analysis is outdated. Due to lack of better sources from usda.gov & census.gov, the outdated data will be used. 

usda.gov -> usda_df: Data reflects the time frame between 2015 - 2019

census.gov -> census_df: Data reflects the time frame between 2019 - 2024 with it's acs5 estimates

census.gov/geodata -> geometry_df: Data is to be reviewed. Seemingly outdated < 2025

## 1. Setup

### 1.1 Import Libraries

In [ ]:
# Importing libraries
import requests
import os
import pandas as pd
import geopandas as gpd

from census import Census
from us import states
from dotenv import load_dotenv

### Load Environments & Declare API Key

In [ ]:

# Some os work with a .env file to ensure security of API Key
load_dotenv()

API_KEY = os.getenv("CENSUS_API_KEY")
print(repr(API_KEY))

## 2. Data Ingestion

### 2.1 Census API Helper

The `census` library was abandoned due to consistent errors. Census data is fetched manually via the ACS 5-year API endpoint.

In [ ]:
# Going to abandon census
# Errors are consistent with the library

# Manually request the information
def fetch_acs5_tracts(fields, state_fips, key, year = 2024): 
    fields_str = ",".join(["NAME"] + list(fields))
    
    url = (
        f"https://api.census.gov/data/{year}/acs/acs5"
        f"?get={fields_str}"
        f"&for=tract:*"
        f"&in=state:{state_fips}"
        f"&key={key}"
    )
    
    response = requests.get(url)
    data = response.json()
    
    return pd.DataFrame(data[1:], columns = data[0])

### Original Census Program

In [ ]:
# REMOVED - CENSUS HAD ISSUES WITH ITS INTERNAL API TO COLLECT REQUIRED INFORMATION

# from us import Census

# c = Census(API_KEY)

# raw_census_api = c.acs5.state_county_tract(
#     fields = (
#         "NAME",
#         "B19013_001E", # Median household income 
#         "B25044_003E", # Owner units, no vehicle 
#         "B17001_002E", # Population below poverty level 
#         "B17001_001E", # Total population 
#         "B25044_010E", # Renter units, no vehicle 
#         "B25044_001E", # Total occupied units 
#         "B03002_003E", # Non-Hispanic white 
#         "B03002_004E", # Black/African American 
#         "B03002_012E", # Hispanic/Latino 
#         "B03002_001E", # Total population 
#         "B22003_002E", # SNAP households 
#         "B22003_001E", # Total households 
#     ), 
#     state_fips = states.WA.fips, 
#     county_fips = Census.ALL,
#     tract = Census.ALL
    
# )

# removed_census_df = pd.DataFrame(raw_census_api)

### Load Raw DataFrames

Three raw sources are loaded: the USDA food access CSV, ACS 5-year Census tract data, and the Washington State census tract shapefile.

In [ ]:
# Creating dataframes

# Using the USDA csv file
raw_usda_df = pd.read_csv("../data/raw/StateAndCountyData.csv")

# Using the Census API Key
raw_census = fetch_acs5_tracts(
    fields = (
        "B19013_001E",
        "B17001_002E",
        "B17001_001E",
        "B25044_003E",
        "B25044_010E",
        "B25044_001E",
        "B03002_003E",
        "B03002_004E",
        "B03002_012E",
        "B03002_001E",
        "B22003_002E",
        "B22003_001E",
    ),
    state_fips = "53",
    key = API_KEY, 
)

raw_census_df = pd.DataFrame(raw_census)


# Using the shp file to get geography with geopandas
raw_geometry_gdf = gpd.read_file("../data/raw/tl_2025_53_tract.shp")

In [ ]:
# Variables List to be merged with raw_usda_df
raw_usda_varaibles_df = pd.read_csv(r"..\data\raw\VariableList.csv")


### API Connectivity Debug

Sanity check that the Census API endpoint responds correctly before proceeding.

Original Purpose - Check API Key Integrity to ensure information is received

In [ ]:
# Running into an issue
# Going to check ip throttle
# Using requests library

url = f"https://api.census.gov/data/2024/acs/acs5?get=NAME,B19013_001E,B25044_003E,B17001_002E,B17001_001E,B25044_010E,B25044_001E,B03002_003E,B03002_004E,B03002_012E,B03002_001E,B22003_002E,B22003_001E&for=tract:*&in=state:53&in=county:*&key={API_KEY}"

response = requests.get(url)

print(response.status_code)
print(response.text[:500])

# Request is working... 

# Check Data Integrity

Check the 4 dataframes to view possible issues

In [ ]:
# USDA Data
print(raw_usda_df.head())
print("=========================================================================================================")

# Describe
print(raw_usda_df.describe())
print("=========================================================================================================")

# Info
print(raw_usda_df.info())

In [ ]:
# Census Data
print(raw_census_df.head())
print("=========================================================================================================")

# Describe
print(raw_census_df.describe())
print("=========================================================================================================")

# Info
print(raw_census_df.info())

In [ ]:
# USDA Data
print(raw_geometry_gdf.head())
print("=========================================================================================================")

# Describe
print(raw_geometry_gdf.describe())
print("=========================================================================================================")

# Info
print(raw_geometry_gdf.info())

In [ ]:
# VariablesList Data
print(raw_usda_varaibles_df.head())
print("========================================================")
print()

print(raw_usda_varaibles_df["Variable_Name"].unique())
print("========================================================")


### Null Value Audit

Checking if any dataframes have null values 

In [ ]:
# Information seems to be in order. OTher things we can check are null values and missing values

print("Number of null values within Census DataFrame")
print(raw_census_df.isna().any().sum())
print()

print("Number of null values within USDA DataFrame")
print(raw_usda_df.isna().any().sum())
print()

print("Number of null values within Geometry DataFrame")
print(raw_geometry_gdf.isna().any().sum())

# Doing Some Data Conversions

### Inspect USDA Column Schema

In [ ]:
print(raw_usda_df.columns)

### Filter USDA Data to Washington State & Merge Variable Labels

In [ ]:
# The USDA csv file needs to be isolated to just Washington state. Then it needs to be isolated further
print(raw_usda_df[raw_usda_df["State"] == "WA"].count())

washington_usda_df = raw_usda_df[raw_usda_df["State"] == "WA"]
# Isolated from ~96k + rows down to ~11k

# Now we need to combine the washington dataset with the given labels
combined_usda_df = pd.merge(
    washington_usda_df, 
    raw_usda_varaibles_df, 
    on = "Variable_Code", 
    how = "left"
)

In [ ]:
print(combined_usda_df.columns)
print("========================================================")
print(combined_usda_df["Category_Code"].unique())
print("========================================================")
print(combined_usda_df["Category_Name"].unique())

# Adjusting Census Data

### Rename Census Columns to Descriptive Names

Original labels do not have "meaningful names"

In [ ]:
# Should have done this earlier. Adjusting the column names to something more meaningful.

column_names = {
    "B19013_001E": "median_income",
    "B17001_002E": "poverty_population",
    "B17001_001E": "poverty_total", # 
    "B25044_003E": "no_vehicle_owner", # No vehicle period
    "B25044_010E": "no_vehicle_renter", # No vehicle, but rents a vehicle
    "B25044_001E": "vehicle_total",
    "B03002_003E": "pop_white",
    "B03002_004E": "pop_black",
    "B03002_012E": "pop_hispanic",
    "B03002_001E": "pop_total",
    "B22003_002E": "snap_households",
    "B22003_001E": "snap_total",
    "NAME":        "tract_name",
    "state":       "state_fips",
    "county":      "county_fips",
    "tract":       "tract_code"
}

raw_census_df = raw_census_df.rename(columns = column_names)

### Initialize Feature-Engineered DataFrame & Build GEOID

In [ ]:
print(raw_census_df)

# Creating a new dataframe to house features after feature engineering
census_df = pd.DataFrame()

census_df['GEOID'] = raw_census_df['state_fips'] + raw_census_df['county_fips'] + raw_census_df['tract_code']

### Identify & Convert Numeric Columns

In [ ]:
# Function will be defined to make pd.to_numeric easier
def get_numeric_columns(dataframe): 
    # Grabs all columns into a temporary dataframe
    df_cols = dataframe.select_dtypes(include = ['str']).columns
    # Defines where the convertible columns will go
    convertible = []
    
    # For each column
    for column in df_cols:
        # Try to convert the column to a numeric value, then add that column to the list 
        try: 
            pd.to_numeric(dataframe[column], errors = 'raise')
            convertible.append(column)
        # Otherwise, raise and catch an error and skip past this iteration
        except (ValueError, TypeError): 
            continue
        
    return convertible

In [ ]:
# Checking which dataframe values can be converted to numeric values if necessary
convertible_columns = get_numeric_columns(raw_census_df)
print(convertible_columns)

for col in convertible_columns: 
    raw_census_df[col] = pd.to_numeric(raw_census_df[col], errors = 'coerce')
    
print(raw_census_df.info())


### Feature Engineering — Derived Statistics

In [ ]:
# Columns are now converted. Build out statistics
census_df['median_income'] = raw_census_df['median_income']
census_df['poverty_percentage'] = raw_census_df['poverty_population'] / raw_census_df['poverty_total']
census_df['no_vehicle_percentage'] = (raw_census_df['no_vehicle_owner'] + raw_census_df['no_vehicle_renter']) / raw_census_df['vehicle_total']
census_df['snap_percentage'] = raw_census_df['snap_households'] / raw_census_df['snap_total']
census_df['percent_black'] = raw_census_df['pop_black'] / raw_census_df['pop_total']
census_df['percent_hispanic'] = raw_census_df['pop_hispanic'] / raw_census_df['pop_total']
census_df['percent_white'] = raw_census_df['pop_white'] / raw_census_df['pop_total']

census_df = census_df.sort_values(by=['median_income', 'poverty_percentage'], ascending = False)

In [ ]:
print(census_df.head())

### Outlier Detection & Cleaning

In [ ]:
#Check for outliers
print(census_df.isna().any().sum())

census_df.dropna(inplace = True)
print(census_df.isna().any().sum())

print(census_df.sort_values(by="median_income"))


In [ ]:
# Some other outliers that are going to mess with the data
census_df = census_df[census_df['median_income'] >= 0]
print(census_df.sort_values(by = 'median_income'))

# Manipulate Remaining DataFrames

Two other dataframes to manipulate: USDA & geometry

In [ ]:
print(combined_usda_df.Variable_Name.unique())
print()

print(raw_geometry_gdf.head())
print()

# There are many labels given to the combined USDA dataframe. Since focus is on poverty, only 

<ArrowStringArray>
[                 'Population, low access to store, 2015',
                  'Population, low access to store, 2019',
   'Population, low access to store (% change), 2015 -19',
              'Population, low access to store (%), 2015',
              'Population, low access to store (%), 2019',
                 'Low income & low access to store, 2015',
                 'Low income & low access to store, 2019',
 'Low income & low access to store (% change), 2015 - 19',
             'Low income & low access to store (%), 2015',
             'Low income & low access to store (%), 2019',
 ...
                            'WIC-authorized stores, 2022',
              'WIC-authorized stores (% change), 2016-22',
                  'WIC-authorized stores/1,000 pop, 2016',
                  'WIC-authorized stores/1,000 pop, 2022',
    'WIC-authorized stores/1,000 pop (% change), 2016-22',
                   'Soda sales tax, retail stores, 2014*',
                         'Soda s

In [33]:
print(combined_usda_df.head())

      FIPS State County          Variable_Code        Value  \
0  53001.0    WA  Adams          LACCESS_POP15  7916.703702   
1  53001.0    WA  Adams          LACCESS_POP19  7811.836680   
2  53001.0    WA  Adams  PCH_LACCESS_POP_15_19    -1.324630   
3  53001.0    WA  Adams      PCT_LACCESS_POP15    42.272018   
4  53001.0    WA  Adams      PCT_LACCESS_POP19    41.712070   

                                       Variable_Name  \
0              Population, low access to store, 2015   
1              Population, low access to store, 2019   
2  Population, low access to store (% change), 20...   
3          Population, low access to store (%), 2015   
4          Population, low access to store (%), 2019   

                       Category_Name Category_Code Subcategory_Name     Units  
0  Access and Proximity to Foodstore        ACCESS          Overall     Count  
1  Access and Proximity to Foodstore        ACCESS          Overall     Count  
2  Access and Proximity to Foodstore        

In [35]:
cols_to_drop = ['Variable_Code', "Category_Code"]

print(combined_usda_df.drop(columns = cols_to_drop).head())

      FIPS State County        Value  \
0  53001.0    WA  Adams  7916.703702   
1  53001.0    WA  Adams  7811.836680   
2  53001.0    WA  Adams    -1.324630   
3  53001.0    WA  Adams    42.272018   
4  53001.0    WA  Adams    41.712070   

                                       Variable_Name  \
0              Population, low access to store, 2015   
1              Population, low access to store, 2019   
2  Population, low access to store (% change), 20...   
3          Population, low access to store (%), 2015   
4          Population, low access to store (%), 2019   

                       Category_Name Subcategory_Name     Units  
0  Access and Proximity to Foodstore          Overall     Count  
1  Access and Proximity to Foodstore          Overall     Count  
2  Access and Proximity to Foodstore          Overall  % change  
3  Access and Proximity to Foodstore          Overall   Percent  
4  Access and Proximity to Foodstore          Overall   Percent  


In [76]:
# Flags to search for within Variable Names
# print(combined_usda_df.Category_Name.unique())

# kept_categories = ['Access and Proximity to Foodstore', 'Socioeconomic Characteristics']
# usda_kept = combined_usda_df.loc[combined_usda_df['Category_Name'].isin(kept_categories)]

# print(len(usda_kept))

#print(usda_kept.Variable_Name.unique())
#print(usda_kept[usda_kept['Variable_Name'].str.contains('Low income & low access', case = False, na = False)])

# print(combined_usda_df[combined_usda_df['Variable_Name'].str.contains("low income & low access", case = False, na = False)])

# Key things to search for
list_of_keywords = ['poverty', 'low income & low access', 'snap', 'population, low access', 'children, low access']

def find_vars(keywords, dataframe, column): 
    key_values = []
    
    for keyword in keywords: 
        found_values = dataframe[dataframe[column].str.contains(keyword, case = False, na = False)][column].unique()
        key_values.extend(found_values)
        
    return key_values

In [79]:
print(find_vars(list_of_keywords, combined_usda_df, 'Variable_Name'))

['Poverty rate, 2021', 'Deep poverty rate, 2021', 'Persistent-poverty counties, 2017-21', 'Child poverty rate, 2021', 'Deep child poverty rate, 2021', 'Low income & low access to store, 2015', 'Low income & low access to store, 2019', 'Low income & low access to store (% change), 2015 - 19', 'Low income & low access to store (%), 2015', 'Low income & low access to store (%), 2019', 'SNAP households, low access to store, 2015', 'SNAP households, low access to store, 2019', 'SNAP households, low access to store (% change), 2015 - 19', 'SNAP households, low access to store (%), 2015', 'SNAP households, low access to store (%), 2019', 'SNAP redemptions/SNAP-authorized stores, 2017', 'SNAP redemptions/SNAP-authorized stores, 2023', 'SNAP redemptions/SNAP-authorized stores (% change), 2017-23', 'SNAP participants (% pop), 2017*', 'SNAP participants (% pop), 2022*', 'SNAP participants (change % pop), 2017-22*', 'SNAP benefits per capita, 2017', 'SNAP benefits per capita, 2022', 'SNAP benefits